### Feature Engineering

- One Hot Encoding (document-level vector)
- Bag of Words using CountVectorizer
- TF-IDF using TfidfVectorizer

In [1]:
# One Hot Encoding
from pathlib import Path
import json

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

data_path = Path('data/amazon_reviews_preprocessed.csv')
mapping_path = Path('data/amazon_reviews_vocabulary_mapping.json')
summary_path = Path('data/amazon_reviews_vocabulary.csv')

for required_path in [data_path, mapping_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

df = pd.read_csv(data_path)
if 'review_text_final' not in df.columns:
    raise KeyError("Expected column 'review_text_final' in the preprocessed dataset.")

texts = df['review_text_final'].fillna('').astype(str)
texts = texts[texts.str.strip() != '']

with mapping_path.open(encoding='utf-8') as f:
    vocabulary_mapping = {str(word): int(index) for word, index in json.load(f).items()}

feature_names = [word for word, _ in sorted(vocabulary_mapping.items(), key=lambda item: item[1])]

if summary_path.exists():
    vocabulary_summary_df = pd.read_csv(summary_path)
    preview_terms = [
        str(word)
        for word in vocabulary_summary_df['word'].astype(str).head(10).tolist()
        if str(word) in vocabulary_mapping
    ]
else:
    preview_terms = feature_names[:10]

preview_term_indices = [vocabulary_mapping[term] for term in preview_terms]

ohe_vectorizer = CountVectorizer(
    vocabulary=vocabulary_mapping,
    binary=True,
    token_pattern=r'(?u)\b\w+\b',
)
ohe_matrix = ohe_vectorizer.fit_transform(texts)

ohe_document_frequency_df = (
    pd.DataFrame(
        {
            'word': feature_names,
            'document_count': (ohe_matrix > 0).sum(axis=0).A1,
        }
    )
    .sort_values(by='document_count', ascending=False)
    .reset_index(drop=True)
)

ohe_preview_df = pd.DataFrame(
    ohe_matrix[:5, preview_term_indices].toarray(),
    columns=preview_terms,
    index=texts.index[:5],
)

print(f"OHE matrix shape: {ohe_matrix.shape}")
print("Top 10 words by document presence:")
print(ohe_document_frequency_df.head(10).to_string(index=False))
print(f"Preview terms: {preview_terms}")

ohe_preview_df


OHE matrix shape: (688, 3251)
Top 10 words by document presence:
   word  document_count
   good             399
  sound             388
quality             387
battery             355
product             219
   bass             216
earbuds             212
  price             200
    use             196
  clear             188
Preview terms: ['good', 'quality', 'sound', 'battery', 'earbuds', 'product', 'phone', 'bud', 'use', 'bass']


,good,quality,sound,battery,earbuds,product,phone,bud,use,bass
0,1,0,0,1,0,0,1,0,0,0
1,1,0,0,1,0,0,1,0,0,0
2,0,0,0,0,0,0,1,0,0,0
3,0,1,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0


In [2]:
# Bag of words using Countvectorizer
if 'texts' not in globals() or 'vocabulary_mapping' not in globals():
    raise RuntimeError("Run the One Hot Encoding cell first to load the dataset and fixed vocabulary.")

bow_vectorizer = CountVectorizer(
    vocabulary=vocabulary_mapping,
    token_pattern=r'(?u)\b\w+\b',
)
bow_matrix = bow_vectorizer.fit_transform(texts)

bow_frequency_df = (
    pd.DataFrame(
        {
            'word': feature_names,
            'count': bow_matrix.sum(axis=0).A1,
        }
    )
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

bow_preview_df = pd.DataFrame(
    bow_matrix[:5, preview_term_indices].toarray(),
    columns=preview_terms,
    index=texts.index[:5],
)

print(f"BoW matrix shape: {bow_matrix.shape}")
print("Top 10 words by total count:")
print(bow_frequency_df.head(10).to_string(index=False))
print(f"Preview terms: {preview_terms}")

bow_preview_df


BoW matrix shape: (688, 3251)
Top 10 words by total count:
   word  count
   good    759
quality    683
  sound    622
battery    499
earbuds    470
product    319
  phone    318
    bud    275
    use    272
   bass    257
Preview terms: ['good', 'quality', 'sound', 'battery', 'earbuds', 'product', 'phone', 'bud', 'use', 'bass']


,good,quality,sound,battery,earbuds,product,phone,bud,use,bass
0,2,0,0,1,0,0,1,0,0,0
1,1,0,0,1,0,0,3,0,0,0
2,0,0,0,0,0,0,2,0,0,0
3,0,1,0,3,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0


In [3]:
# TF-IDF using TFidFVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

if 'texts' not in globals() or 'vocabulary_mapping' not in globals():
    raise RuntimeError("Run the One Hot Encoding cell first to load the dataset and fixed vocabulary.")

tfidf_vectorizer = TfidfVectorizer(
    vocabulary=vocabulary_mapping,
    token_pattern=r'(?u)\b\w+\b',
)
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

tfidf_score_df = (
    pd.DataFrame(
        {
            'word': feature_names,
            'mean_tfidf': tfidf_matrix.mean(axis=0).A1,
        }
    )
    .sort_values(by='mean_tfidf', ascending=False)
    .reset_index(drop=True)
)

tfidf_preview_df = pd.DataFrame(
    tfidf_matrix[:5, preview_term_indices].toarray(),
    columns=preview_terms,
    index=texts.index[:5],
).round(4)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print("Top 10 words by mean TF-IDF score:")
print(tfidf_score_df.head(10).to_string(index=False))
print(f"Preview terms: {preview_terms}")

tfidf_preview_df


TF-IDF matrix shape: (688, 3251)
Top 10 words by mean TF-IDF score:
   word  mean_tfidf
   good    0.094410
quality    0.064033
product    0.051896
  sound    0.050201
battery    0.047278
  phone    0.037715
earbuds    0.036034
  great    0.031549
   bass    0.031289
  price    0.030468
Preview terms: ['good', 'quality', 'sound', 'battery', 'earbuds', 'product', 'phone', 'bud', 'use', 'bass']


,good,quality,sound,battery,earbuds,product,phone,bud,use,bass
0,0.1788,0.0000,0.0,0.0962,0.0,0.0,0.1341,0.0,0.0,0.0
1,0.0677,0.0000,0.0,0.0728,0.0,0.0,0.3047,0.0,0.0,0.0
2,0.0000,0.0000,0.0,0.0000,0.0,0.0,0.2719,0.0,0.0,0.0
3,0.0000,0.0457,0.0,0.1445,0.0,0.0,0.0000,0.0,0.0,0.0
4,0.0000,0.0000,0.0,0.0000,0.0,0.0,0.1949,0.0,0.0,0.0


### Comparison Analysis

- Create a comparison table for OHE, BoW, and TF-IDF
- Explain words are most important in TF-IDF and why common words receive lower weight

In [4]:
# Comparison analysis
from IPython.display import display

required_objects = [
    'ohe_matrix',
    'bow_matrix',
    'tfidf_matrix',
    'feature_names',
    'tfidf_vectorizer',
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        f"Run the OHE, BoW, and TF-IDF cells first. Missing objects: {missing_objects}"
    )

comparison_table_df = pd.DataFrame(
    [
        {
            'Method': 'One Hot Encoding',
            'Value Type': 'Binary (0 or 1)',
            'What It Captures': 'Whether a word appears in a review',
            'Strength': 'Simple document-level presence signal',
            'Limitation': 'Ignores repetition and word importance',
            'Matrix Shape': str(ohe_matrix.shape),
        },
        {
            'Method': 'Bag of Words',
            'Value Type': 'Integer count',
            'What It Captures': 'How many times a word appears in a review',
            'Strength': 'Preserves term frequency inside each review',
            'Limitation': 'Common words can dominate because no rarity weighting is used',
            'Matrix Shape': str(bow_matrix.shape),
        },
        {
            'Method': 'TF-IDF',
            'Value Type': 'Weighted float score',
            'What It Captures': 'Importance of a word in a review relative to the whole corpus',
            'Strength': 'Highlights informative words and downweights common ones',
            'Limitation': 'Still ignores word order and semantic meaning',
            'Matrix Shape': str(tfidf_matrix.shape),
        },
    ]
)

idf_analysis_df = pd.DataFrame(
    {
        'word': feature_names,
        'document_count': (ohe_matrix > 0).sum(axis=0).A1,
        'bow_count': bow_matrix.sum(axis=0).A1,
        'idf': tfidf_vectorizer.idf_,
        'mean_tfidf': tfidf_matrix.mean(axis=0).A1,
    }
)

top_tfidf_words_df = (
    idf_analysis_df.sort_values(by='mean_tfidf', ascending=False)
    .reset_index(drop=True)
    .head(10)
)

lowest_idf_words_df = (
    idf_analysis_df.sort_values(
        by=['idf', 'document_count', 'bow_count'],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
    .head(10)
)

important_words = ', '.join(top_tfidf_words_df['word'].head(5).tolist())
common_words = ', '.join(lowest_idf_words_df['word'].head(5).tolist())
overlap_words = sorted(
    set(top_tfidf_words_df['word']).intersection(lowest_idf_words_df['word'])
)
overlap_preview = ', '.join(overlap_words[:5])

print('Comparison Table:')
display(comparison_table_df)

print('Top 10 words by mean TF-IDF score:')
display(top_tfidf_words_df)

print('Words with the lowest IDF values (most common across reviews):')
display(lowest_idf_words_df)

print('Explanation:')
print(
    f'TF-IDF gives high importance to words like {important_words} because they have strong overall impact after combining term frequency and inverse document frequency across the review corpus.'
)
print(
    f'Words like {common_words} have the lowest IDF values, which means they are common across many reviews. TF-IDF penalizes these words compared with plain word counts, but does not force their importance to zero.'
)
if overlap_words:
    print(
        f'There is some overlap between the two lists, such as {overlap_preview}. This is not a contradiction: TF-IDF is TF multiplied by IDF, so a word can still rank high overall if its term frequency is strong enough, even after receiving an IDF penalty for being common.'
    )


Comparison Table:


,Method,Value Type,What It Captures,Strength,Limitation,Matrix Shape
0,One Hot Encoding,Binary (0 or 1),Whether a word appears in a review,Simple document-level presence signal,Ignores repetition and word importance,"(688, 3251)"
1,Bag of Words,Integer count,How many times a word appears in a review,Preserves term frequency inside each review,Common words can dominate because no rarity we...,"(688, 3251)"
2,TF-IDF,Weighted float score,Importance of a word in a review relative to t...,Highlights informative words and downweights c...,Still ignores word order and semantic meaning,"(688, 3251)"


Top 10 words by mean TF-IDF score:


,word,document_count,bow_count,idf,mean_tfidf
0,good,399,759,1.543777,0.094410
1,quality,387,683,1.574236,0.064033
2,product,219,319,2.141614,0.051896
3,sound,388,622,1.571662,0.050201
4,battery,355,499,1.660311,0.047278
5,phone,184,318,2.314885,0.037715
6,earbuds,212,470,2.173949,0.036034
7,great,183,237,2.320306,0.031549
8,bass,216,257,2.155344,0.031289
9,price,200,241,2.231936,0.030468


Words with the lowest IDF values (most common across reviews):


,word,document_count,bow_count,idf,mean_tfidf
0,good,399,759,1.543777,0.094410
1,sound,388,622,1.571662,0.050201
2,quality,387,683,1.574236,0.064033
3,battery,355,499,1.660311,0.047278
4,product,219,319,2.141614,0.051896
5,bass,216,257,2.155344,0.031289
6,earbuds,212,470,2.173949,0.036034
7,price,200,241,2.231936,0.030468
8,use,196,272,2.252038,0.028987
9,clear,188,228,2.293494,0.026352


Explanation:
TF-IDF gives high importance to words like good, quality, product, sound, battery because they have strong overall impact after combining term frequency and inverse document frequency across the review corpus.
Words like good, sound, quality, battery, product have the lowest IDF values, which means they are common across many reviews. TF-IDF penalizes these words compared with plain word counts, but does not force their importance to zero.
There is some overlap between the two lists, such as bass, battery, earbuds, good, price. This is not a contradiction: TF-IDF is TF multiplied by IDF, so a word can still rank high overall if its term frequency is strong enough, even after receiving an IDF penalty for being common.


### Sparse Matrix Analysis
- Print shape of matrices, calculate sparsity (percentage of zeros)
- Explain why sparse matrices are inefficient for large-scale systems

In [5]:
# Sparse matrix analysis
from IPython.display import display

required_matrices = {
    'One Hot Encoding': 'ohe_matrix',
    'Bag of Words': 'bow_matrix',
    'TF-IDF': 'tfidf_matrix',
}

missing_matrices = [
    name for name, var_name in required_matrices.items() if var_name not in globals()
]
if missing_matrices:
    raise RuntimeError(
        f"Run the OHE, BoW, and TF-IDF cells first. Missing matrices: {missing_matrices}"
    )

sparse_analysis_rows = []

for method_name, var_name in required_matrices.items():
    matrix = globals()[var_name]
    rows, cols = matrix.shape
    total_cells = rows * cols
    non_zero = matrix.nnz
    zero_count = total_cells - non_zero
    density_percent = (non_zero / total_cells) * 100
    sparsity_percent = (zero_count / total_cells) * 100
    dense_memory_mb = matrix.toarray().nbytes / (1024 ** 2)
    sparse_memory_mb = (
        matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    ) / (1024 ** 2)

    sparse_analysis_rows.append(
        {
            'Method': method_name,
            'Shape': f'{rows} x {cols}',
            'Non-zero Values': non_zero,
            'Zero Values': zero_count,
            'Density (%)': round(density_percent, 4),
            'Sparsity (%)': round(sparsity_percent, 4),
            'Sparse Memory (MB)': round(sparse_memory_mb, 4),
            'Dense Memory (MB)': round(dense_memory_mb, 4),
        }
    )

sparse_analysis_df = pd.DataFrame(sparse_analysis_rows)

print('Sparse Matrix Analysis Table:')
display(sparse_analysis_df)

print('Explanation:')
print(
    'These matrices are highly sparse because each review contains only a small fraction of the full vocabulary, so most cells are zero.'
)
print(
    'Sparse storage is much more memory-efficient than dense storage, but very large-scale systems can still become inefficient because vocabulary size keeps growing, indexing overhead increases, and many matrix operations become slower across millions of rows and features.'
)
print(
    'So the practical problem is not that sparse format is worse than dense format; it is that high-dimensional sparse representations still create storage, compute, and scaling overhead in real NLP pipelines.'
)


Sparse Matrix Analysis Table:


,Method,Shape,Non-zero Values,Zero Values,Density (%),Sparsity (%),Sparse Memory (MB),Dense Memory (MB)
0,One Hot Encoding,688 x 3251,29399,2207289,1.3144,98.6856,0.3391,17.0646
1,Bag of Words,688 x 3251,29399,2207289,1.3144,98.6856,0.3391,17.0646
2,TF-IDF,688 x 3251,29399,2207289,1.3144,98.6856,0.3391,17.0646


Explanation:
These matrices are highly sparse because each review contains only a small fraction of the full vocabulary, so most cells are zero.
Sparse storage is much more memory-efficient than dense storage, but very large-scale systems can still become inefficient because vocabulary size keeps growing, indexing overhead increases, and many matrix operations become slower across millions of rows and features.
So the practical problem is not that sparse format is worse than dense format; it is that high-dimensional sparse representations still create storage, compute, and scaling overhead in real NLP pipelines.


- Why Bag of Words fails in understanding semantic meaning (example: similar meaning words)
- When to use Bag of Words and TF-IDF in industry
- Limitations of TF-IDF in real applications

### Answers

**1. Why Bag of Words fails in understanding semantic meaning**

Bag of Words only counts words and ignores meaning, context, and word order. Because of that, two sentences can have similar meaning but very different word vectors if they use different words. For example, *good phone* and *excellent mobile* express similar sentiment, but Bag of Words treats `good`, `phone`, `excellent`, and `mobile` as unrelated features. It also cannot understand negation properly. For example, *this product is good* and *this product is not good* may look very similar even though their meanings are opposite.

**2. When to use Bag of Words and TF-IDF in industry**

Bag of Words is useful when you want a simple baseline model, fast experimentation, or interpretable word-count features. It is often used in small text-classification tasks such as spam detection, review classification, or topic tagging when speed and simplicity matter more than deep language understanding. TF-IDF is preferred when you want to reduce the effect of very common words and highlight more informative terms. In industry, TF-IDF is commonly used in document search, ticket categorization, FAQ matching, recommendation support, and other retrieval or classification tasks where word importance matters more than raw counts.

**3. Limitations of TF-IDF in real applications**

TF-IDF still does not understand semantics, synonyms, sarcasm, or context. Words with similar meanings are treated as different features, and word order is ignored. It also depends heavily on the vocabulary seen during training, so unseen words in new data are not represented well. In large real-world systems, TF-IDF can create very high-dimensional sparse matrices, which increase memory and computation cost. Another limitation is that TF-IDF measures importance statistically, not semantically, so a word can receive a high score even if it does not fully represent the intent of the sentence.
